# 1. Setup

**Approach 2 · Crop-Conditioned Pair Scoring.** Compare Frozen DINOv2 + Cross-Crop Disease Prototypes with the same recipe plus a compact crop-conditioned disease head. The primary comparison is the two raw 333-way results. The notebook also reports the calibrated rows as a separate ranking diagnostic.

Open the notebook inside the project repository in VS Code. Use `DEBUG=False` and Run All for complete development, refit and evaluation. Debug mode runs a short train-only check. The raw unseen targets of 40% and 50% are marked in the comparison CSV.

In [ ]:
DEBUG = True

from datetime import datetime, timezone
from pathlib import Path
import copy
from dataclasses import dataclass
import hashlib
import json
import sys

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
BATCH_SIZE, EPOCHS = (8, 1) if DEBUG else (16, 20)
HEAD_LR, WEIGHT_DECAY = 1e-4, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
PROTOTYPE_WEIGHT, PROTOTYPE_TEMPERATURE = 0.15, 0.10
CONDITIONAL_WIDTH, RESIDUAL_LIMIT, CONDITIONAL_WEIGHT = 64, 2.0, 0.5
GAMMAS = [value / 2 for value in range(25)]
RUN_NAME = "approach2_crop_conditioned" + ("_debug" if DEBUG else "")
MODEL_NAME = "approach2_crop_conditioned"
B0_UNSEEN_REFERENCE = 9.60
RAW_UNSEEN_TARGET = round(B0_UNSEEN_REFERENCE * 1.30, 2)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
OFFICIAL_DIR = RESULTS_DIR / "official" / "pair_identity_order"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError("Training CSV or image directory is missing.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the shared 323/333 evaluation API.")
print(f"PyTorch {torch.__version__} · {DEVICE} · debug={DEBUG} · run {RUN_ID}")

# 2. Data

Development uses the team train/validation split. Hold out ten complete training pairs from the development fit data and prototype bank. Use five for tuning and five for an independent check. Both methods use the same images and pairs. Full refit uses every training CSV row. Official metadata enters only in Evaluation.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


def proxy_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Proxy validation needs seen and held-out images.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")

settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW", "schedule": "CosineAnnealingLR", "label_smoothing": SMOOTHING,
    "backbone_frozen": True, "decor_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT,
    "temperature": TEMPERATURE, "blur_probability": BLUR_PROB, "blur_sigma": BLUR_SIGMA,
    "gamma_grid": GAMMAS, "gamma_selection": "proxy harmonic mean, seen top1, lower gamma",
    "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}


@dataclass(frozen=True)
class Trial:
    code: str
    prediction_names: tuple
    development_dir: Path
    final_dir: Path
    checkpoint: Path
    lock: Path
    settings: dict


def make_trial(code, crop_conditioning):
    final_dir = RESULTS_DIR / "final" / code
    return Trial(code, (f"{code}_raw", f"{code}_calibrated"), RESULTS_DIR / "development" / code,
                 final_dir, CHECKPOINT_DIR / code / "model.pt", final_dir / "selection_lock.json",
                 {**settings, "prototype_weight": PROTOTYPE_WEIGHT,
                  "prototype_temperature": PROTOTYPE_TEMPERATURE, "crop_conditioning": crop_conditioning,
                  "conditional_width": CONDITIONAL_WIDTH, "residual_limit": RESIDUAL_LIMIT,
                  "conditional_weight": CONDITIONAL_WEIGHT})


prototype = make_trial("Prototype", False)
conditional = make_trial("Conditional", True)
trials = (conditional,) if DEBUG else (prototype, conditional)

# 3. Model

Both recipes use frozen DINOv2 ViT-B/14, the A3 concept heads and losses, and cross-crop disease prototypes. The conditional recipe adds a 64-dimensional image–crop interaction whose disease-logit residual starts at zero and is bounded to ±2.

For a candidate crop `c` and disease `d`, score `log P(c | image) + log P(d | image, c)`. Normalize each conditioned disease prediction over all 175 diseases. Candidate crop IDs come from the candidate table. The model uses the same rule for every candidate.

The residual learns from the true crop and disease labels of fit images. The auxiliary conditional loss updates the residual branch. The original crop and disease heads have the same objectives in both recipes. This is a lightweight adaptation of the conditional factorization in Wu et al. [ICCV 2025](https://openaccess.thecvf.com/content/ICCV2025/html/Wu_A_Conditional_Probability_Framework_for_Compositional_Zero-shot_Learning_ICCV_2025_paper.html)

In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


def augment_blur(images, generator):
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return images
    selected = selected.to(images.device)
    augmented = images.clone()
    augmented[selected] = TF.gaussian_blur(images[selected], kernel_size=[5, 5],
                                           sigma=[BLUR_SIGMA, BLUR_SIGMA])
    return augmented


class CrossCropPrototypes:
    """Train-only disease prototypes. Every supporting crop receives equal weight."""

    def __init__(self, pair_means, pairs):
        self.means = pair_means.detach()
        self.diseases = pairs[:, 1]
        self.host_counts = torch.bincount(self.diseases, minlength=NUM_DISEASES)
        self.host_index = torch.zeros((NUM_CROPS, NUM_DISEASES), dtype=torch.long, device=pair_means.device)
        self.host_index[pairs[:, 0], self.diseases] = torch.arange(1, len(pairs) + 1, device=pair_means.device)

    def loss(self, features, crop, disease, disease_head):
        local_index = self.host_index[crop]
        other_counts = self.host_counts - local_index.gt(0).long()
        available = other_counts.gt(0)
        eligible = available.gather(1, disease[:, None]).squeeze(1) & available.sum(1).gt(1)
        if not eligible.any():
            return features.new_zeros(()), 0

        # A linear projection commutes with averaging, so project each host mean once.
        projected = F.linear(self.means, disease_head.weight)
        totals = projected.new_zeros((NUM_DISEASES, projected.shape[1])).index_add(
            0, self.diseases, projected)
        by_host = F.pad(projected, (0, 0, 1, 0))[local_index[eligible]]
        external = (totals - by_host) / other_counts[eligible].clamp_min(1).unsqueeze(2)
        external = F.normalize(external - external.mean(-1, keepdim=True), dim=-1)
        query = F.normalize(center_logits(F.linear(features[eligible], disease_head.weight)), dim=1)
        scores = (external * query[:, None, :]).sum(-1) / PROTOTYPE_TEMPERATURE
        scores = scores.masked_fill(~available[eligible], -torch.inf)
        return F.cross_entropy(scores, disease[eligible]), int(eligible.sum())


@torch.no_grad()
def build_prototypes(backbone, fit_frame):
    loader = DataLoader(PlantDataset(fit_frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    sums = torch.zeros((len(train_pairs), backbone.num_features), device=DEVICE)
    counts = torch.zeros(len(train_pairs), dtype=torch.long, device=DEVICE)
    for images, _, _, pair_rows in loader:
        pair_rows = pair_rows.to(DEVICE)
        sums.index_add_(0, pair_rows, backbone(images.to(DEVICE)).float())
        counts.index_add_(0, pair_rows, torch.ones_like(pair_rows))
    active = counts > 0
    if counts.sum().item() != len(fit_frame):
        raise RuntimeError("Prototype pass did not visit the entire fit partition.")
    return CrossCropPrototypes(sums[active] / counts[active, None], train_pairs.to(DEVICE)[active])


class CropDiseaseResidual(nn.Module):
    """Shared visual residual conditioned on a crop concept."""

    def __init__(self, width):
        super().__init__()
        self.visual = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, CONDITIONAL_WIDTH))
        self.crop = nn.Embedding(NUM_CROPS, CONDITIONAL_WIDTH)
        self.out = nn.Linear(CONDITIONAL_WIDTH, NUM_DISEASES, bias=False)
        nn.init.zeros_(self.out.weight)

    def forward(self, features, crops):
        interaction = self.visual(features).tanh() * self.crop(crops).tanh()
        return RESIDUAL_LIMIT * self.out(interaction).tanh()

    def candidate_residuals(self, features, crops):
        interaction = self.visual(features).tanh()[:, None, :] * self.crop(crops).tanh()[None, :, :]
        return RESIDUAL_LIMIT * self.out(interaction).tanh()


def score_pairs(features, crop_logits, disease_logits, pairs, conditional=None):
    if conditional is None:
        return pair_scores(crop_logits, disease_logits, pairs)

    pairs = pairs.to(crop_logits.device)
    crop_ids, crop_slots = pairs[:, 0].unique(sorted=True, return_inverse=True)
    residuals = conditional.candidate_residuals(features, crop_ids)
    disease_by_crop = F.log_softmax(disease_logits.float()[:, None, :] + residuals, dim=-1)
    crop_scores = F.log_softmax(crop_logits.float(), dim=-1)[:, pairs[:, 0]]
    return crop_scores + disease_by_crop[:, crop_slots, pairs[:, 1]]

# 4. Training

Train both recipes from the same initialization and on the same batches. Use raw harmonic mean on train-only proxy validation to pick epochs and choose each optional penalty γ on the train-only tune split. Refit both from pretrained DINOv2 on the full training CSV for their selected durations. Lock both checkpoints and all scoring rules before official evaluation.

In [ ]:
def selection_key(row):
    return (row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"])


def snapshot_tail(tail):
    return {name: value.detach().cpu().clone() for name, value in tail.state_dict().items()}


def proxy_metrics(seen_group, proxy_group, fit_mask, gamma=0.0):
    seen_s, seen_y, _ = seen_group
    proxy_s, proxy_y, disease_pred = proxy_group
    seen_s, proxy_s = (scores - gamma * fit_mask for scores in (seen_s, proxy_s))
    seen = top_k_accuracy(seen_s, seen_y, 1)
    proxy = top_k_accuracy(proxy_s, proxy_y, 1)
    held_scores = proxy_s.masked_fill(fit_mask[None, :], -torch.inf)
    winner, held_winner = proxy_s.argmax(dim=1), held_scores.argmax(dim=1)
    blocked = held_winner.eq(proxy_y) & fit_mask[winner]
    disease_top1 = 100 * disease_pred.eq(train_pairs[proxy_y, 1]).float().mean().item()
    return {"gamma": gamma, "seen_top1": seen, "proxy_top1": proxy, "harmonic_top1": harmonic_mean(seen, proxy),
            "proxy_disease_head_top1": round(disease_top1, 2),
            "proxy_blocked_by_fit_pairs_pct": round(100 * blocked.float().mean().item(), 2)}


def choose_penalty(tune_groups, check_groups, fit_mask, trial):
    grid = [proxy_metrics(*tune_groups, fit_mask, gamma) for gamma in GAMMAS]
    best = max(grid, key=lambda row: (row["harmonic_top1"], row["seen_top1"], -row["gamma"]))
    gamma = best["gamma"]
    check = [proxy_metrics(*check_groups, fit_mask, value) for value in (0.0, gamma)]
    pd.DataFrame(grid).to_csv(trial.development_dir / "calibration_grid.csv", index=False)
    pd.DataFrame(check, index=trial.prediction_names).rename_axis("model").to_csv(
        trial.development_dir / "independent_proxy_check.csv")
    return gamma, check


def read_final_lock(trial):
    if not trial.lock.is_file() or not trial.checkpoint.is_file():
        raise FileNotFoundError("Complete full refit first.")
    saved = json.loads(trial.lock.read_text(encoding="utf-8"))
    if saved["settings"] != trial.settings or saved["checkpoint_sha256"] != sha256_file(trial.checkpoint):
        raise RuntimeError("Checkpoint or settings changed after selection.")
    if saved["seen_pairs"] != train_pairs.tolist():
        raise RuntimeError("Training candidate identities changed after selection.")
    return saved


def heads_from(model, trial):
    heads = nn.ModuleDict({"crop": copy.deepcopy(model.crop_head), "disease": copy.deepcopy(model.disease_head)})
    if trial.settings["crop_conditioning"]:
        heads["conditional"] = CropDiseaseResidual(model.backbone.num_features).to(DEVICE)
    return heads


@torch.no_grad()
def cache_features(backbone, frame):
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    outputs = [(backbone(images.to(DEVICE)).cpu(), pairs) for images, _, _, pairs in loader]
    return tuple(torch.cat(list(parts)) for parts in zip(*outputs))


@torch.no_grad()
def score_cache(heads, cache):
    features, labels = cache
    features = features.to(DEVICE)
    crop, disease = heads["crop"](features), heads["disease"](features)
    residual = heads["conditional"] if "conditional" in heads else None
    return score_pairs(features, crop, disease, train_pairs, residual).cpu(), labels, disease.argmax(1).cpu()


def fit_heads(model, loader, epochs, fit_mask=None, tune=None):
    fit_frame = loader.dataset.frame
    if fit_mask is not None:
        fit_frame = fit_frame.loc[fit_frame.pair_row.isin(fit_mask.nonzero(as_tuple=True)[0].tolist())]
    prototypes = build_prototypes(model.backbone, fit_frame)
    units = {trial.code: heads_from(model, trial) for trial in trials}
    prototype_weights = {trial.code: trial.settings["prototype_weight"] for trial in trials}
    optimizers = {code: torch.optim.AdamW(heads.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
                  for code, heads in units.items()}
    schedules = {code: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
                 for code, opt in optimizers.items()}
    fit_device = fit_mask.to(DEVICE) if fit_mask is not None else None
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    history, best = [], {}

    for epoch in range(1, max(epochs.values()) + 1):
        active = [code for code, limit in epochs.items() if epoch <= limit]
        totals = {code: [0., 0., 0, 0, 0.] for code in active}
        examples = 0
        for step, (images, crop, disease, pair_rows) in enumerate(loader, 1):
            images = augment_blur(images.to(DEVICE), blur_rng)
            if fit_device is not None:
                keep = fit_device[pair_rows.to(DEVICE)]
                if not keep.any():
                    continue
                images, crop, disease = images[keep], crop.to(DEVICE)[keep], disease.to(DEVICE)[keep]
            else:
                crop, disease = crop.to(DEVICE), disease.to(DEVICE)
            with torch.no_grad():
                features = model.backbone(images)
            examples += len(images)
            for code in active:
                heads, optimizer = units[code], optimizers[code]
                crop_logits, disease_logits = heads["crop"](features), heads["disease"](features)
                crop_repr, disease_repr = center_logits(crop_logits), center_logits(disease_logits)
                supcon, anchors = cross_crop_supcon(disease_repr, crop, disease)
                loss = criterion(crop_logits, crop) + criterion(disease_logits, disease)
                loss = loss + DECOR_WEIGHT * decorrelation(crop_repr, disease_repr) + SUPCON_WEIGHT * supcon
                proto_loss, proto_anchors = features.new_zeros(()), 0
                if prototype_weights[code]:
                    proto_loss, proto_anchors = prototypes.loss(features, crop, disease, heads["disease"])
                    loss = loss + prototype_weights[code] * proto_loss
                conditional_loss = features.new_zeros(())
                if "conditional" in heads:
                    refined = disease_logits.detach() + heads["conditional"](features, crop)
                    conditional_loss = criterion(refined, disease)
                    loss = loss + CONDITIONAL_WEIGHT * conditional_loss
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Non-finite training loss for {code}.")
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                record = totals[code]
                record[0] += loss.item() * len(images)
                record[1] += proto_loss.item() * proto_anchors
                record[2] += int(anchors)
                record[3] += proto_anchors
                record[4] += conditional_loss.item() * len(images)
            if step == 1 or step % 50 == 0 or step == len(loader):
                print(f"Epoch {epoch}/{max(epochs.values())} · batch {step}/{len(loader)}", flush=True)
        if examples != len(fit_frame):
            raise RuntimeError("Training did not process every fit image exactly once.")

        for code in active:
            schedules[code].step()
            total, proto, anchors, proto_anchors, conditional = totals[code]
            row = {"recipe": code, "epoch": epoch, "images_trained": examples, "train_loss": total / examples,
                   "supcon_anchor_fraction": anchors / examples, "prototype_anchor_fraction": proto_anchors / examples,
                   "prototype_loss": proto / max(proto_anchors, 1), "conditional_disease_ce": conditional / examples}
            if tune is not None:
                metrics = proxy_metrics(*(score_cache(units[code], group) for group in tune), fit_mask)
                row.update({key: value for key, value in metrics.items() if key != "gamma"})
                if code not in best or selection_key(row) > selection_key(best[code]["metrics"]):
                    best[code] = {"metrics": row.copy(), "weights": snapshot_tail(units[code])}
            history.append(row)
        print(pd.DataFrame(history[-len(active):]).to_string(index=False), flush=True)
    return units, history, best


def run_development():
    team.set_seed()
    train, val, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Development loaders must use only the training CSV.")
    train.pin_memory = False
    model = MultiTaskNet(pretrained=True).to(DEVICE).eval()
    fit_mask, frames = (None, {}) if DEBUG else proxy_split(train.dataset.frame, val.dataset.frame)
    tune = None if DEBUG else [cache_features(model.backbone, frames[key]) for key in ("tune_seen", "tune_proxy")]
    units, history, best = fit_heads(model, train, {trial.code: EPOCHS for trial in trials}, fit_mask, tune)
    for trial in trials:
        trial.development_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([row for row in history if row["recipe"] == trial.code]).to_csv(
            trial.development_dir / "training_history.csv", index=False)
    if DEBUG:
        return {}

    check = [cache_features(model.backbone, frames[key]) for key in ("check_seen", "check_proxy")]
    selections = {}
    for trial in trials:
        heads = units[trial.code]
        heads.load_state_dict(best[trial.code]["weights"])
        tune_scores, check_scores = ([score_cache(heads, group) for group in split] for split in (tune, check))
        gamma, check_metrics = choose_penalty(tune_scores, check_scores, fit_mask, trial)
        selections[trial.code] = {
            "selection": best[trial.code]["metrics"], "gamma": gamma, "independent_proxy_check": check_metrics,
            "selection_tune_pair_rows": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
            "independent_check_pair_rows": sorted(frames["check_proxy"].pair_row.unique().tolist()),
        }
        atomic_json(trial.development_dir / "epoch_selection.json", selections[trial.code])
    return selections


developments = run_development()

In [ ]:
def refit_full_train():
    if any(trial.lock.is_file() for trial in trials):
        raise FileExistsError("This run is already locked. Rerun Setup for a new run.")
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development.")
    epochs = {code: row["selection"]["epoch"] for code, row in developments.items()}
    if set(epochs) != {trial.code for trial in trials} or not all(1 <= epoch <= EPOCHS for epoch in epochs.values()):
        raise ValueError("One selected epoch count per recipe is required.")

    team.set_seed()
    model = MultiTaskNet(pretrained=True).to(DEVICE).eval()
    train_transform, _ = get_transforms("imagenet")
    loader = DataLoader(PlantDataset(train_full, train_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=0, drop_last=False,
                        generator=torch.Generator().manual_seed(team.SEED))
    units, history, _ = fit_heads(model, loader, epochs)
    runs = {}
    model.cpu()
    for trial in trials:
        code = trial.code
        selected = developments[code]
        model.crop_head.load_state_dict(units[code]["crop"].state_dict())
        model.disease_head.load_state_dict(units[code]["disease"].state_dict())
        model.conditional = (copy.deepcopy(units[code]["conditional"]).cpu()
                             if "conditional" in units[code] else None)
        refit = {"initialization": "original_pretrained_dinov2_fresh_heads", "data": "all_training_csv_rows",
                 "images_per_epoch": len(train_full), "training_pairs": len(train_pairs),
                 "epochs": epochs[code], "cosine_t_max": EPOCHS}
        metadata = {"settings": trial.settings, "recipe": code, "refit": refit,
                    "seen_pairs": train_pairs.tolist(), "selection": selected["selection"],
                    "prediction_rules": dict(zip(trial.prediction_names, (0.0, selected["gamma"])))}
        trial.final_dir.mkdir(parents=True, exist_ok=True)
        trial.checkpoint.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame([row for row in history if row["recipe"] == code]).to_csv(
            trial.final_dir / "refit_history.csv", index=False)
        atomic_save(trial.checkpoint, {"state_dict": model.state_dict(), **metadata})
        locked = {**selected, **metadata, "checkpoint_sha256": sha256_file(trial.checkpoint),
                  "checkpoint_bytes": trial.checkpoint.stat().st_size}
        atomic_json(trial.lock, locked)
        runs[code] = locked
    return runs


runs = {} if DEBUG else refit_full_train()

# 5. Evaluation

Verify both locks before reading the ten official unseen crop/disease identities. Combine them with the 323 training-derived pairs. Read filenames and generate both recipes' raw and calibrated predictions over all 333 candidates before joining per-image targets. Compute seen, unseen and overall results plus harmonic mean. The true-crop oracle column is a diagnostic computed after labels are available. It ranks the subset of official candidates with the true crop using raw scores.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    generator = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=generator)
    mean, std = (torch.tensor(values).view(1, 3, 1, 1) for values in team.NORM_STATS["imagenet"])
    pair_indices = pairs.to(DEVICE)
    scores, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        features = model.backbone(images.to(DEVICE))
        crop, disease = model.crop_head(features), model.disease_head(features)
        batch = score_pairs(features, crop, disease, pair_indices, getattr(model, "conditional", None)).cpu()
        if batch.shape != (len(images), len(pairs)) or not torch.isfinite(batch).all():
            raise ValueError("Invalid raw pair scores.")
        scores.append(batch)
        concepts.append(torch.stack((crop.argmax(dim=1), disease.argmax(dim=1)), dim=1).cpu())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(scores), torch.cat(concepts)


def predict_trial(trial, run, pairs, seen_names, unseen_names):
    rules = run["prediction_rules"]
    if rules != dict(zip(trial.prediction_names, (0.0, run["gamma"]))) or run["gamma"] not in GAMMAS:
        raise ValueError("Final inference rules differ from the development lock.")
    saved = torch.load(trial.checkpoint, map_location="cpu", weights_only=True)
    for key in ("settings", "prediction_rules", "seen_pairs", "selection", "refit"):
        if saved[key] != run[key]:
            raise ValueError(f"Checkpoint and selection lock differ for {key}.")
    model = MultiTaskNet(pretrained=False)
    if trial.settings["crop_conditioning"]:
        model.conditional = CropDiseaseResidual(model.backbone.num_features)
    model.load_state_dict(saved["state_dict"], strict=True)
    model.to(DEVICE).eval().requires_grad_(False)

    seen_scores, seen_heads = predict_official(model, pairs, seen_names, "seen")
    unseen_scores, unseen_heads = predict_official(model, pairs, unseen_names, "unseen")
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for name, transform in PERTURBATIONS.items():
            perturbed[name], _ = predict_official(model, pairs, seen_names, name, transform)
    # Materialize every prediction for both frozen rules before loading per-image targets.
    predictions = {}
    for name, gamma in rules.items():
        penalty = gamma * membership
        predictions[name] = {
            "seen": top_predictions(seen_scores - penalty),
            "unseen": top_predictions(unseen_scores - penalty),
            "robust": {key: (scores - penalty).topk(1, dim=1).indices
                       for key, scores in perturbed.items()},
        }
    seen_only = top_predictions(seen_scores[:, :boundary])
    unseen_only = top_predictions(unseen_scores[:, boundary:], offset=boundary)
    return predictions, seen_only, unseen_only, unseen_scores, seen_heads, unseen_heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, seen_only, unseen_only, unseen_scores, seen_heads, unseen_heads = forward_pass
    rules = run["prediction_rules"]
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_head_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    only_correct = unseen_only[1].squeeze(1).eq(y_unseen)
    reports, ranking, per_pair, robustness = [], [], [], []
    same_crop_candidates = pairs[:, 0].eq(pairs[y_unseen, :1])
    crop_oracle = unseen_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
    oracle_top1 = prediction_accuracy(crop_oracle, y_unseen)
    margins = unseen_scores.gather(1, y_unseen[:, None]).squeeze(1) - unseen_scores[:, :boundary].amax(dim=1)
    for name, prediction in predictions.items():
        metrics = {"model": name, "recipe": run["recipe"], "gamma": rules[name], "epoch": run["selection"]["epoch"],
                   "candidates": len(pairs), **head_top1, "raw_true_crop_candidate_top1": oracle_top1}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        if name.endswith("_raw"):
            metrics["historical_B0_unseen_top1"] = B0_UNSEEN_REFERENCE
            metrics["relative_unseen_gain_vs_B0_pct"] = round(
                100 * (metrics["unseen_top1"] / B0_UNSEEN_REFERENCE - 1), 2)
            metrics["meets_30pct_relative_target"] = metrics["unseen_top1"] >= RAW_UNSEEN_TARGET
            metrics["meets_raw_40pct"] = metrics["unseen_top1"] >= 40.0
            metrics["meets_raw_50pct"] = metrics["unseen_top1"] >= 50.0
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"model": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(margins.median()) + rules[name]})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"model": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"model": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
    restricted = [
        {"split": split, "candidates": count, **{f"top{k}": prediction_accuracy(pred[k], labels) for k in (1, 5)}}
        for split, count, pred, labels in (("seen_only", boundary, seen_only, y_seen),
                                          ("unseen_only", len(pairs) - boundary, unseen_only, y_unseen))
    ]
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def official_evaluation():
    if DEBUG or not runs or any(read_final_lock(trial) != runs[trial.code] for trial in trials):
        raise RuntimeError("Complete both full refits and lock their settings before official evaluation.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    seen_names, unseen_names = map(official_filenames, paths)
    if set(seen_names) & set(unseen_names) or set(train_full.image) & (set(seen_names) | set(unseen_names)):
        raise ValueError("Image names overlap between training and evaluation splits.")
    forward_passes = {trial.code: predict_trial(trial, runs[trial.code], pairs, seen_names, unseen_names)
                      for trial in trials}
    # Both checkpoints and all four scoring rules have predicted before labels are joined.
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], seen_names, rows, seen=True)
    y_unseen = official_targets(paths[1], unseen_names, rows, seen=False)
    summary = {"selection": runs, "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    winners = {}
    for code, predictions in forward_passes.items():
        report, seen_winners = score_trial(runs[code], pairs, predictions, y_seen, y_unseen)
        winners.update(seen_winners)
        for key, value in report.items():
            for row in value:
                row.setdefault("recipe", code)
            summary.setdefault(key, []).extend(value)
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()
if evaluation is None:
    print("Debug logs:", trials[0].development_dir)

# 6. Save

The Approach 3C-style comparison CSV and JSON report `Prototype_raw` and `Conditional_raw` alongside their calibrated rows. Review raw unseen Top-1, seen Top-1, harmonic mean, raw true-crop accuracy and the same-crop/other-crop blocked counts. The run folders contain both full `model.pt` checkpoints, refit histories, confusion matrices and robustness reports.

In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / f"{MODEL_NAME}_comparison.csv", index=False)
    for name in ("restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['model']}.json", row)
    atomic_json(OFFICIAL_DIR / f"{MODEL_NAME}_comparison.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    display(table[["model", "gamma", "seen_top1", "unseen_top1", "overall_top1", "harmonic_mean",
                   "unseen_disease_head_top1", "raw_true_crop_candidate_top1",
                   "meets_raw_40pct", "meets_raw_50pct"]])
    for trial in trials:
        print(trial.code, "checkpoint:", trial.checkpoint.resolve())
    print("Results:", OFFICIAL_DIR.resolve())